In [1]:
#!pip install biopython>=1.83 pandas>=2.0.0 matplotlib>=3.7.0 seaborn>=0.12.0 pyarrow>=15.0.0 openpyxl>=3.1.0

In [2]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2
import os
import numpy as np
import scipy as sp
import pandas as pd
import networkx as nx
import random
import matplotlib.pyplot as plt
# import warnings filter
from warnings import simplefilter
# ignore all future warnings
simplefilter(action='ignore', category=FutureWarning)
from IPython.core.display import HTML
HTML('<style>.container { width:95% !important; }</style><style>.output_png {display: table-cell;text-align: center;vertical-align: middle;}</style>')

In [ ]:

from amr.data_collection import (
    add_venue_vectors,
    build_antibiotic_pathogen_query,
    expand_synonyms,
    fetch_pmids_by_date_and_keywords,
    fetch_pmids_intervals,
    fetch_pub_metadata,
    get_metadata,
    merge_with_existing,
    parse_pubdate,
    plot_venue_vector,
    prepare_antibiotic_pubmed_query,
    retrieve_pmids,
    search_pmids,
    update_metadata_cache,
)


---

In [4]:
antibiotics_df = pd.read_excel("antibiotic_list.xlsx")
antibiotics_list = antibiotics_df['Antibiotic'].values
print(f'#antibiotics: {len(antibiotics_list)}')

data_df = pd.read_parquet("data.parquet")
pathogens_list = data_df[data_df['n_causation']>=3]['CanonicalSpecies'].values
print(f'#pathogens: {len(pathogens_list)}')

bacteria_synonym_df = pd.read_csv("bacteria_synonym_lookup.csv")
print(f'#synonyms: {len(bacteria_synonym_df)}')

#antibiotics: 245
#pathogens: 773
#synonyms: 24932


In [5]:
if False:
    pathogens_list = ['Staphylococcus argenteus',
    'Staphylococcus aureus', 'Staphylococcus auricularis',
    'Staphylococcus capitis', 'Staphylococcus caprae',
    'Staphylococcus cohnii', 'Staphylococcus condimenti',
    'Staphylococcus epidermidis', 'Staphylococcus gallinarum',
    'Staphylococcus haemolyticus', 'Staphylococcus hominis',
    'Staphylococcus hyicus', 'Staphylococcus intermedius',
    'Staphylococcus lugdunensis', 'Staphylococcus pasteuri',
    'Staphylococcus pettenkoferi', 'Staphylococcus pseudintermedius',
    'Staphylococcus saccharolyticus', 'Staphylococcus saprophyticus',
    'Staphylococcus schleiferi', 'Staphylococcus simulans',
    'Staphylococcus warneri', 'Staphylococcus xylosus',
    'Neisseria animaloris', 'Neisseria bacilliformis',
    'Neisseria canis', 'Neisseria cinerea', 'Neisseria elongata',
    'Neisseria flava', 'Neisseria flavescens', 'Neisseria gonorrhoeae',
    'Neisseria lactamica', 'Neisseria macacae',
    'Neisseria meningitidis', 'Neisseria mucosa', 'Neisseria perflava',
    'Neisseria sicca', 'Neisseria subflava', 'Neisseria weaveri']

In [6]:
if False:
    pathogens_list = ["Escherichia coli", "Staphylococcus aureus", "Neisseria canis"]
    antibiotics_list = ["Piperacillin/tazobactam", "Cefteram-pivoxil"]

In [ ]:
parquet_path = "pmids_results.parquet"

df = fetch_pmids_intervals(
    pathogens=pathogens_list,
    antibiotics=antibiotics_list,
    bacteria_synonym_df=bacteria_synonym_df,
    min_date="1975/01/01",
    max_date="2024/12/31",
    year_gap=5,
    extra_keywords=[
        "(resistant OR resistance OR nonsusceptible OR non-susceptible)"
    ],
    parquet_path=parquet_path
)
#"NOT (susceptible OR susceptibility OR sensitive OR sensitivity)"
merge_with_existing(df, parquet_path=parquet_path)

In [ ]:
df = pd.read_parquet("pmids_results.parquet")
df

In [5]:
df_with_vecs, venue_index = add_venue_vectors(df)
venue_index_df = pd.Series({pos: venue for venue, pos in venue_index.items()}, name="venue").to_frame()
venue_index_df.to_parquet('venue_index.parquet', index=False)
df_with_vecs.to_parquet('pmids_results_venues.parquet', index=False)
df_with_vecs

,pathogen,antibiotic,start_date,end_date,pmids,count,venue_histogram,venue_vector
0,Abiotrophia defectiva,Spectinomycin,1975/01/01,1979/12/31,[],0,{},"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,Abiotrophia defectiva,Spectinomycin,1980/01/01,1984/12/31,[],0,{},"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,Abiotrophia defectiva,Spectinomycin,1985/01/01,1989/12/31,[],0,{},"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,Abiotrophia defectiva,Spectinomycin,1990/01/01,1994/12/31,[],0,{},"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,Abiotrophia defectiva,Spectinomycin,1995/01/01,1999/12/31,[],0,{},"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
...,...,...,...,...,...,...,...,...
1893845,Yokenella regensburgei,Trimethoprim,2000/01/01,2004/12/31,[],0,{},"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1893846,Yokenella regensburgei,Trimethoprim,2005/01/01,2009/12/31,[],0,{},"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1893847,Yokenella regensburgei,Trimethoprim,2010/01/01,2014/12/31,[],0,{},"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1893848,Yokenella regensburgei,Trimethoprim,2015/01/01,2019/12/31,[],0,{},"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


---